In [65]:
from utils.database_utils import generate_database_and_retriever
from neo4j import GraphDatabase
import json

URI = "bolt://localhost:7687"
AUTH = ("neo4j", "123456789")


def get_graph_context(graph_driver, retrieved_nodes):
    """
    Takes the output from your MultiVectorRetriever and
    fetches a 2-hop neighborhood for each node.
    """

    all_knowledge_blocks = []
    with graph_driver.session() as session:
        connections = {}
        entities = set({})

        for full_node_params in retrieved_nodes:
            params = {
                "name": full_node_params["name"],
                "label": full_node_params["type"],
                "description": full_node_params["description"],
            }
            # Assuming retrieved_nodes are Document objects from your retriever
            # We pull the name or ID from the metadata

            query = """
            MATCH (source)
            WHERE source.name = $name 
            AND $label IN labels(source) 
            AND source.description = $description

            MATCH path = (source)-[*1..2]-(neighbor)
            WHERE source <> neighbor
            AND NONE(lbl IN labels(neighbor) WHERE lbl IN ['text', 'image', 'table', 'document'])
            AND NONE(rel IN relationships(path) WHERE type(rel) = 'BELONGS_TO')

            WITH source, neighbor, collect(path) AS paths
            UNWIND paths as p
            UNWIND relationships(p) as rel
            RETURN DISTINCT collect({
                subject: startNode(rel).name,
                subject_description: startNode(rel).description, 
                predicate: type(rel),
                object: endNode(rel).name,
                object_description: endNode(rel).description
            }) AS connection_paths
            """

            def format_rel(rel_type):
                return rel_type.replace("_", " ").lower()

            results = session.run(query, **params)

            # Collect the results:

            for record in results:
                for connection in record["connection_paths"]:
                    subject = connection["subject"]
                    subject_desc = connection["subject_description"]
                    predicate = connection["predicate"]
                    object_ = connection["object"]
                    object_desc = connection["object_description"]

                    # Adding entities
                    entities.add((subject, subject_desc))
                    entities.add((object_, object_desc))

                    # Adding connections
                    if (subject, subject_desc) not in connections:
                        connections[(subject, subject_desc)] = {}

                    if predicate not in connections[(subject, subject_desc)]:
                        connections[(subject, subject_desc)][predicate] = set([])

                    connections[(subject, subject_desc)][predicate].add(
                        (object_, object_desc)
                    )

        entities_narrative = []
        for entity in entities:
            entities_narrative.append(
                f"\t'{entity[0]}': {entity[1] or 'No description.'}"
            )

        connections_narrative = []
        for subject, subject_desc in connections.keys():
            for predicate, object_object_desc in connections[
                (subject, subject_desc)
            ].items():
                for object_, object_desc in object_object_desc:
                    connections_narrative.append(
                        f"\t'{subject}' {format_rel(predicate)} '{object_}'"
                    )
            # Create a dense, summarized block
        summary_block = (
            f"ENTITIES:\n{',\n'.join(entities_narrative)}:\n",
            f"RELATIONSHIPS:\n{',\n'.join(connections_narrative)}\n",
        )
        all_knowledge_blocks.append("\n".join(summary_block))

    return "\n".join(all_knowledge_blocks)


In [75]:
from langchain_ollama import OllamaLLM
from langchain_core.messages import SystemMessage, HumanMessage


def summarize_graph_context(nodes_context, model_name="gemma3:latest"):
    system_prompt = """

        You are an expert in knowledge synthesis and technical summarization.

        Your task is to transform structured knowledge (entities + relationships) into a dense, high-signal summary optimized for retrieval in a RAG system.

        ---

        ### OBJECTIVE:
        Generate a compact, information-rich representation that:
        - Preserves all critical technical facts and metrics
        - Consolidates duplicate or conflicting values (prefer most consistent or repeated signals)
        - Removes redundancy and noise
        - Filters out invalid, weak, or illogical relationships
        - Clearly separates model structure, inputs/outputs, and performance

        ---

        ### INPUT:
        You will receive:
        1. ENTITIES: Named concepts with descriptions
        2. RELATIONSHIPS: Triplets connecting entities

        ---

        ### PROCESSING RULES:
        - Deduplicate metrics (e.g., multiple F1/accuracy values → summarize as ranges or most representative values)
        - Ignore contradictory or after reasoning low-confidence relationships unless strongly supported
        - Normalize synonyms (e.g., "optimal shade", "optimal threshold", "cutoff value" → one concept)

        ---

        ### OUTPUT FORMAT:

        1. **Dense Summary (5–8 sentences max)**  
        - Highly compressed, technical narrative  
        - Must include: model types, inputs, outputs, key metrics, thresholds, and interpretability approach  

        2. **Structured Key Insights (bullet points)**  
        - Max 8 bullets  
        - Each bullet = one atomic, high-value insight  
        - Prefer normalized terminology and grouped metrics  

        ---

        ### STYLE:
        - Technical and precise
        - High signal-to-noise ratio
        - No repetition
        - No explanations or commentary
        - No hallucinated connections

        ---

        ### CONSTRAINT:
        Only use information grounded in the provided entities and relationships.
        Do not infer beyond the data unless necessary for normalization.
        Return only the final summary and bullet points.

    """
    messages = [
        SystemMessage(content=system_prompt),
        HumanMessage(content=f"{nodes_context}"),
    ]
    model = OllamaLLM(model=model_name)
    return model.invoke(messages)

In [3]:
def parse_nodes(nodes_retrieved):
    nodes_parsed = []
    for node in nodes_retrieved:
        nodes_parsed.append(json.loads(node.decode("utf-8")))
    return nodes_parsed

# Retrievers

You have to initiate 4 different retrievers: 
 - For regular rag
 - For nodes related to the search query
 - For mid level communities
 - For global level communities

In [4]:
regular_rag_retriever = retriever = generate_database_and_retriever(
    main_folder="./localdb"
)

nodes_retriever = generate_database_and_retriever(main_folder="./localdb/node_db")

mid_level_retriever = generate_database_and_retriever(
    main_folder="./localdb/mid_communities"
)

global_level_retriever = generate_database_and_retriever(
    main_folder="./localdb/global_communities"
)

In [5]:
docs_retrieved = retriever.invoke("liniar regression model")

nodes_retrieved = nodes_retriever.invoke("liniar regression model")
mid_level_communities_retrieved = mid_level_retriever.invoke("model")
global_level_communities_retrieved = global_level_retriever.invoke("model")


In [6]:
nodes_parsed = parse_nodes(nodes_retrieved)

In [66]:
with GraphDatabase.driver(URI, auth=AUTH) as driver:
    nodes_context = get_graph_context(driver, nodes_parsed)

In [77]:
description = summarize_graph_context(nodes_context)

In [78]:
print(description)

Dense Summary:
The system employs a decision tree model and a linear regression model for analysis, utilizing a training dataset of 260 samples and 121 samples for evaluation respectively. Both models leverage input features including collagen fiber length, collagen aggregate number, volatile organic compound peak number, average intensity, porosity, and fiber orientation. The decision tree model achieved an F1 score of 0.782 and an accuracy of 0.901, while the linear regression model attained an F1 score of 0.812 and an accuracy of 0.8. An optimal cutoff value of 0.48 was identified via the ROC curve for both models. Model interpretability is assessed through the decision tree and linear regression models.

Structured Key Insights:
*   Model Types: Decision Tree, Linear Regression
*   Key Metrics: F1 Score (0.782, 0.812), Accuracy (0.901, 0.8)
*   Optimal Threshold: 0.48
*   Training Data: 260 samples, 121 samples
*   Input Features: Collagen fiber length, collagen aggregate number, v